---

**TAREAS 6-9**
---


</div>

---

| **Universitario** | Vismark Abner Choque Cachi |
|------------------|---------------------------|
| **Materia**      | Deep Learning        |
| **Docente**      | Ph.D. Moises Martin Silva Choque|
| **Sigla**        | DAT-264                   |
---
---



## **Llenado hacia Atras BACKFILL SIN LIBRERIAS**

In [3]:
# Librerias genericas
import pandas as pd
import numpy as np

In [4]:
# Crearemos un dataframe
data = {'fecha': pd.to_datetime(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05']),
        'ventas': [100, np.nan, 120, np.nan, 150]}
df = pd.DataFrame(data)

print("DataFrame original:")
print(df)

DataFrame original:
       fecha  ventas
0 2025-01-01   100.0
1 2025-01-02     NaN
2 2025-01-03   120.0
3 2025-01-04     NaN
4 2025-01-05   150.0


In [ ]:
# Se usa la imputación de llenado hacia atrás (bfill)
df['ventas'] = df['ventas'].bfill()
#Para cada NaN, lo reemplaza con el valor válido que encuentra hacia abajo en la misma columna.

print("DataFrame con llenado hacia atrás:")
print(df)

DataFrame con llenado hacia atrás:
       fecha  ventas
0 2025-01-01   100.0
1 2025-01-02   120.0
2 2025-01-03   120.0
3 2025-01-04   150.0
4 2025-01-05   150.0


# **TAREA(6)**
**BACKFILL SIN LIBRERIAS**

In [6]:
# DataFrame anterior
data = {
    'fecha': ['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05'],
    'ventas': [100, None, 120, None, 150]
}

# Mostrar DataFrame antes del backfill
print("DataFrame antes de backfill:")


for i in range(len(data['fecha'])):
    print(f"{data['fecha'][i]}  {data['ventas'][i]}")

# Funcion de backfill
def backfill(column):
    # recorrer de atrás hacia adelante
    for i in range(len(column)-2, -1, -1):
        if column[i] is None:
            column[i] = column[i+1]

backfill(data['ventas'])

# Mostrar DataFrame despues del backfill
print("\nDataFrame despues de backfill:")
for i in range(len(data['fecha'])):
    print(f"{data['fecha'][i]}  {data['ventas'][i]}")


DataFrame antes de backfill:
2025-01-01  100
2025-01-02  None
2025-01-03  120
2025-01-04  None
2025-01-05  150

DataFrame despues de backfill:
2025-01-01  100
2025-01-02  120
2025-01-03  120
2025-01-04  150
2025-01-05  150


---


# **IMPUTACION KNN**

In [7]:
def knn_imputer(df, n_neighbors=2):

    # Se crea una copia del DataFrame original para no modificar los datos fuente
    df_copy = df.copy()

    # Paso 1. Normalizacion de los datos (Min-Max Scaling).
    # La funcion calcula los valores minimos y maximos de cada columna.
      # La normalizacion se realiza porque la distancia euclidiana es muy sensible a la escala de las variables.
    # Si una columna tiene un rango muy grande respecto a otra, dominara en el calculo de la distancia.
    # La transformacion se hace con la formula: x' = (x - min(x)) / (max(x) - min(x)),
    # lo cual lleva todos los valores al rango [0, 1].
    df_min = df_copy.min()
    df_max = df_copy.max()
    df_normalized = (df_copy - df_min) / (df_max - df_min)

    # Paso 2. Iterar a traves de cada celda con valor faltante.
    # np.argwhere devuelve las coordenadas (fila, columna) donde existe un NaN en el DataFrame normalizado.
    for row_pos, col_pos in np.argwhere(df_normalized.isna().values):
        # Se obtienen los indices reales de fila y columna usando los nombres en el DataFrame
        row_idx = df_normalized.index[row_pos]       # indice de fila real
        col_name = df_normalized.columns[col_pos]    # nombre de la columna real

        # Se prepara una lista para almacenar las distancias de esta fila hacia todas las demas
        distances = []

        # Paso 3. Calcular la distancia euclidiana a todas las demas filas
        for other_pos, other_idx in enumerate(df_normalized.index):
            # Se omite la comparacion con la misma fila
            if other_idx == row_idx:
                continue

            # Paso 4. Determinar las columnas validas para el calculo de distancia
            # Una columna es valida si no contiene NaN ni en la fila actual ni en la fila candidata
            valid_cols = ~df_normalized.loc[row_idx].isna() & ~df_normalized.loc[other_idx].isna()

            # Si existen columnas validas, se calcula la distancia euclidiana entre ambas filas
            if np.any(valid_cols):
                dist = np.linalg.norm(
                    df_normalized.loc[row_idx, valid_cols] -
                    df_normalized.loc[other_idx, valid_cols]
                )
                # Se almacena la distancia junto con el indice de la fila candidata
                distances.append((dist, other_idx))

        # Paso 5. Seleccionar los k vecinos mas cercanos
        # Se ordenan las distancias en orden ascendente
        distances.sort(key=lambda x: x[0])
        # Se extraen los indices de las filas vecinas mas cercanas
        neighbors = [idx for _, idx in distances[:n_neighbors]]

        # Paso 6. Imputar el valor faltante
        # Se calcula la media de los valores en la columna objetivo (col_name)
        # de las filas vecinas seleccionadas
        imputed_value = df_copy.loc[neighbors, col_name].mean()

        # En caso de que todos los vecinos tambien tengan NaN en esa columna,
        # se usa la media global de la columna como respaldo
        if np.isnan(imputed_value):
            imputed_value = df_copy[col_name].mean()

        # Se asigna el valor imputado en la celda correspondiente
        df_copy.loc[row_idx, col_name] = imputed_value

    # Finalmente, la funcion devuelve el DataFrame con los valores imputados
    return df_copy


In [8]:
# DataFrame de ejemplo
data = {'feature_A': [10, 20, np.nan, 40, 50],
        'feature_B': [1.5, 2.5, 3.5, np.nan, 5.5],
        'feature_C': [100, 200, 300, 400, np.nan]}
df = pd.DataFrame(data)

print("DataFrame original:")
print(df)

# Aplicar la funciOn
df_imputed_manual = knn_imputer(df, n_neighbors=2)

print("\nDataFrame después de la imputaciOn manual:")
print(df_imputed_manual)

DataFrame original:
   feature_A  feature_B  feature_C
0       10.0        1.5      100.0
1       20.0        2.5      200.0
2        NaN        3.5      300.0
3       40.0        NaN      400.0
4       50.0        5.5        NaN

DataFrame después de la imputaciOn manual:
   feature_A  feature_B  feature_C
0       10.0        1.5      100.0
1       20.0        2.5      200.0
2       30.0        3.5      300.0
3       40.0        4.5      400.0
4       50.0        5.5      350.0


---

# **TAREA (7)**
realiza la imputacion knn con libreria sklearn

In [9]:
import pandas as pd
import numpy as np
import tensorflow as tf

def imputador_knn_tf(df, vecinos=2):
    df_copia = df.copy()
    df_min = df_copia.min()
    df_max = df_copia.max()
    df_normalizado = (df_copia - df_min) / (df_max - df_min)

    for fila_pos, col_pos in np.argwhere(df_normalizado.isna().values):
        indice_fila = df_normalizado.index[fila_pos]
        nombre_columna = df_normalizado.columns[col_pos]

        distancias = []
        for otra_pos, otro_indice in enumerate(df_normalizado.index):
            if otro_indice == indice_fila:
                continue

            columnas_validas = ~df_normalizado.loc[indice_fila].isna() & ~df_normalizado.loc[otro_indice].isna()
            if np.any(columnas_validas):
                a = tf.constant(df_normalizado.loc[indice_fila, columnas_validas].values, dtype=tf.float32)
                b = tf.constant(df_normalizado.loc[otro_indice, columnas_validas].values, dtype=tf.float32)
                distancia = tf.norm(a - b).numpy()
                distancias.append((distancia, otro_indice))

        # Ordenar distancias manualmente sin lambda
        for i in range(len(distancias)):
            for j in range(i+1, len(distancias)):
                if distancias[i][0] > distancias[j][0]:
                    distancias[i], distancias[j] = distancias[j], distancias[i]

        vecinos_cercanos = [indice for distancia, indice in distancias[:vecinos]]
        valor_imputado = df_copia.loc[vecinos_cercanos, nombre_columna].mean()
        if np.isnan(valor_imputado):
            valor_imputado = df_copia[nombre_columna].mean()
        df_copia.loc[indice_fila, nombre_columna] = valor_imputado

    return df_copia

# Ejemplo
datos = {
    'feature_A': [10, 20, np.nan, 40, 50],
    'feature_B': [1.5, 2.5, 3.5, np.nan, 5.5],
    'feature_C': [100, 200, 300, 400, np.nan]
}
df = pd.DataFrame(datos)

print("DataFrame original:")
print(df)

df_imputado = imputador_knn_tf(df, vecinos=2)

print("\nDataFrame después de la imputacion con TensorFlow:")
print(df_imputado)


DataFrame original:
   feature_A  feature_B  feature_C
0       10.0        1.5      100.0
1       20.0        2.5      200.0
2        NaN        3.5      300.0
3       40.0        NaN      400.0
4       50.0        5.5        NaN

DataFrame después de la imputacion con TensorFlow:
   feature_A  feature_B  feature_C
0       10.0        1.5      100.0
1       20.0        2.5      200.0
2       30.0        3.5      300.0
3       40.0        4.5      400.0
4       50.0        5.5      350.0


# **WINZORIZACION**

In [ ]:
def winsorizar(data, lower_limit=0.10, upper_limit=0.10):
    """
    Realiza la winsorización en una lista de datos numéricos.

    Args:
        data (list): La lista de números a winsorizar.
        lower_limit (float): El percentil inferior para el límite (e.g., 0.10 para el 10%).
        upper_limit (float): El percentil superior para el límite (e.g., 0.10 para el 90%).

    Returns:
        list: La lista de datos winsorizados.
    """
    # Paso 1. Copiar y ordenar los datos para no modificar la lista original
    datos_ordenados = sorted(data)
    n = len(datos_ordenados)

    # Paso 2. Calcular los índices para los límites
    indice_inferior = int(lower_limit * (n - 1))
    indice_superior = int((1 - upper_limit) * (n - 1))

    # Paso 3. Obtener los valores de los límites
    valor_inferior = datos_ordenados[indice_inferior]
    valor_superior = datos_ordenados[indice_superior]

    # Paso 4. Winsorizar los datos
    datos_winsorizados = []
    for valor in data:
        if valor < valor_inferior:
            datos_winsorizados.append(valor_inferior)
        elif valor > valor_superior:
            datos_winsorizados.append(valor_superior)
        else:
            datos_winsorizados.append(valor)

    return datos_winsorizados

# **TAREA (8)**
**Hacer funcionar la winsorizacion**

In [ ]:
from scipy.stats.mstats import winsorize

# Creamos el dataset de prueba
ingresos = [50000, 60000, 75000, 80000, 95000, 110000, 120000, 130000, 250000, 1500000]

print("Datos originales:")
print(ingresos)

# Convertir la lista a un array de NumPy
ingresos_np = np.array(ingresos)

# Aplicar la función de winsorización
ingresos_winsorizados_np = winsorize(ingresos_np, limits=(0.1, 0.1))

print("Datos originales:")
print(ingresos)

print("\nDatos winsorizados (usando scipy):")
print(ingresos_winsorizados_np)

Datos originales:
[50000, 60000, 75000, 80000, 95000, 110000, 120000, 130000, 250000, 1500000]
Datos originales:
[50000, 60000, 75000, 80000, 95000, 110000, 120000, 130000, 250000, 1500000]

Datos winsorizados (usando scipy):
[ 60000  60000  75000  80000  95000 110000 120000 130000 250000 250000]


# **TAREA (9)**
**etiquetar, binarizar con sklearn**

In [13]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

# Datos iniciales
data = [[7, 2, 3], [4, np.nan, 6], [10, 5, 9]]
data_np = np.array(data)

print("Media general de los datos (ignorando NaN):", np.nanmean(data_np))
print("\nMatriz indicando NaN:\n", np.isnan(data_np))
print("\nMediana por columna (ignorando NaN):", np.nanmedian(data_np, axis=0))

# Imputacion de valores faltantes
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
imp_mean.fit(data)
X = [[np.nan, 2, 3], [4, np.nan, 6], [10, np.nan, 9]]
X_imputado = imp_mean.transform(X)
print("\nDatos despues de imputacion con la media:\n", X_imputado)

# Etiquetar variables categoricas
categorical = ['rojo', 'verde', 'azul', 'rojo', 'verde']
label_encoder = LabelEncoder()
categorical_encoded = label_encoder.fit_transform(categorical)
print("\nEtiquetas codificadas:", categorical_encoded)

# Binarizar (One-Hot Encoding)
onehot_encoder = OneHotEncoder(sparse_output=False)
categorical_encoded_reshaped = categorical_encoded.reshape(-1, 1)
categorical_binarizado = onehot_encoder.fit_transform(categorical_encoded_reshaped)

print("\nDatos binarizados (One-Hot Encoding):\n", categorical_binarizado)


Media general de los datos (ignorando NaN): 5.75

Matriz indicando NaN:
 [[False False False]
 [False  True False]
 [False False False]]

Mediana por columna (ignorando NaN): [7.  3.5 6. ]

Datos despues de imputacion con la media:
 [[ 7.   2.   3. ]
 [ 4.   3.5  6. ]
 [10.   3.5  9. ]]

Etiquetas codificadas: [1 2 0 1 2]

Datos binarizados (One-Hot Encoding):
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
